# Problem Statement

### Context

A significant number of hotel bookings are called off due to cancellations or no-shows. The typical reasons for cancellations include change of plans, scheduling conflicts, etc. This is often made easier by the option to do so free of charge or preferably at a low cost which is beneficial to hotel guests but it is a less desirable and possibly revenue-diminishing factor for hotels to deal with. Such losses are particularly high on last-minute cancellations.

The new technologies involving online booking channels have dramatically changed customers’ booking possibilities and behavior. This adds a further dimension to the challenge of how hotels handle cancellations, which are no longer limited to traditional booking and guest characteristics.

The cancellation of bookings impact a hotel on various fronts:
1. Loss of resources (revenue) when the hotel cannot resell the room.
2. Additional costs of distribution channels by increasing commissions or paying for publicity to help sell these rooms.
3. Lowering prices last minute, so the hotel can resell a room, resulting in reducing the profit 
4. Human resources to make arrangements for the guests.

## Objective

The increasing number of cancellations calls for a Machine Learning based solution that can help in predicting which booking is likely to be canceled. INN Hotels Group has a chain of hotels in Portugal, they are facing problems with the high number of booking cancellations and have reached out to your firm for data-driven solutions. You as a data scientist have to analyze the data provided to find which factors have a high influence on booking cancellations, build a predictive model that can predict which booking is going to be canceled in advance, and help in formulating profitable policies for cancellations and refunds. guests.

## Data Description

The data contains the different attributes of customers' booking details. The detailed data dictionary is given below.

**Data Dictionary**
* Booking_ID: the unique identifier of each booking
* no_of_adults: Number of adults
* no_of_children: Number of Children
* no_of_weekend_nights: Number of weekend nights (Saturday or Sunday) the guest stayed or booked to stay at the hotel
* no_of_week_nights: Number of weeknights (Monday to Friday) the guest stayed or booked to stay at the hotel
* type_of_meal_plan: Type of meal plan booked by the customer: Not Selected – No meal plan selected, Meal Plan 1 – Breakfast, Meal Plan 2 – Half board (breakfast and one other meal), Meal Plan 3 – Full board (breakfast, lunch, and dinner)
* required_car_parking_space: Does the customer require a car parking space? (0 - No, 1- Yes)
* room_type_reserved: Type of room reserved by the customer. The values are ciphered (encoded) by INN Hotels Group
* lead_time: Number of days between the date of booking and the arrival date
* arrival_year: Year of arrival date
* arrival_month: Month of arrival date
* arrival_date: Date of the month
* market_segment_type: Market segment designation.
* repeated_guest: Is the customer a repeated guest? (0 - No, 1- Yes)
* no_of_previous_cancellations: Number of previous bookings that were canceled by the customer prior to the current booking
* no_of_previous_bookings_not_canceled: Number of previous bookings not canceled by the customer prior to the current booking
* avg_price_per_room: Average price per day of the reservation; prices of the rooms are dynamic. (in euros)
* no_of_special_requests: Total number of special requests made by the customer (e.g. high floor, view from the room, etc)
* booking_status: Flag indicating if the booking was canceled or not.

# **Importing the necessary libraries**

In [ ]:
# Libraries to help with reading and manipulating data
import pandas as pd
import numpy as np

# libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)
# setting the precision of floating numbers to 5 decimal points
pd.set_option("display.float_format", lambda x: "%.5f" % x)

# Library to split data
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

# To build model for prediction
import statsmodels.api as SM
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

# To tune different models
from sklearn.model_selection import GridSearchCV

# To get diferent metric scores
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    make_scorer
)

# to suppress unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

# **Loading the dataset**

In [ ]:
Data = pd.read_csv('INNHotelsGroup.csv') 

In [ ]:
# copying data to another variable to avoid any changes to original data
data = Data.copy()

## **Data Overview**

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.shape

 #### The dataset has 36275 rows and 19 columns.

### Checking the data types of the columns for the dataset.

In [ ]:
data.info()

* The `Booking ID`, `type_of_meal_plan`, `room_type_reserved`, `market_segment_type`, `booking_status` columns are of *object* type while the rest columns are numeric in nature

### Checking for missing values

* There are no null values in the dataset

### Dropping the duplicate values

In [ ]:
# checking for duplicate values
data.duplicated().sum()

* There are no duplicate values in the data.

#### Dropping the columns with all unique values

In [ ]:
data.Booking_ID.nunique()

* The `Booking_ID` column contains only unique values, Dropping the column.

### Statistical summary of the data

In [ ]:
data.describe().T

1. Number of Adults
The range varies from a minimum of 0 adults to a maximum of 4 adults per booking. Most bookings (25%, 50%, and 75%) typically involve 2 adults.

2. Number of Children
The number of children ranges from 0 to 10. A significant majority of bookings (25%, 50%, and 75%) do not include any children.

3. Number of Weekend Nights
The number of weekend nights ranges from 0 to 7. Many bookings (25%) have no weekend nights, while 50% and 75% of bookings have 1 and 2 weekend nights, respectively.

4. Number of Week Nights
The number of week nights ranges from 0 to 17. While 25% of bookings include 1 week night, 50% have 2 week nights, and 75% have 3 week nights.

5. Required Car Parking Space
The number of required parking spaces ranges from 0 to 1, with the majority of bookings (25%, 50%, and 75%) not requiring any car parking space.

6. Lead Time
The lead time ranges from 0 to 443 days. While 25% of bookings have a lead time of 17 days, 50% have a lead time of 57 days, and 75% have a lead time of 126 days.

7. Arrival Year
The arrival years range from 2017 to 2018, with 25% and 75% of bookings recorded in 2017 and 50% of bookings in 2018.

8. Arrival Month
The arrival months range from January (1) to December (12). The months of May, August, and October represent the 25th, 50th, and 75th percentiles, respectively.

9. Arrival Date
The arrival dates range from the 1st to the 31st of the month. The 8th, 16th, and 23rd days of the month represent the 25th, 50th, and 75th percentiles, respectively.

10. Repeated Guest
The number of repeated guests ranges from 0 to 1, with the majority of bookings (25%, 50%, and 75%) not involving repeated guests.

11. Number of Previous Cancellations
The number of previous cancellations ranges from 0 to 13. Most bookings (25%, 50%, and 75%) have no previous cancellations.

12. Number of Previous Bookings Not Canceled
The number of previous bookings not canceled ranges from 0 to 72. A significant majority of bookings (25%, 50%, and 75%) have no previous bookings not canceled.

13. Average Price per Room
The price per room ranges from 0 to 540 units. The prices at the 25th, 50th, and 75th percentiles are 69.29, 94.50, and 126.00 units, respectively.

14. Number of Special Requests
The number of special requests ranges from 0 to 5. Many bookings (25%) have no special requests, while 50% and 75% of bookings have 0 and 1 special request, respectively.


## <a name='link2'>Exploratory Data Analysis (EDA) Summary</a>

In [ ]:
def histogram_boxplot(data, feature, figsize=(15, 10), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (15,10))
    kde: whether to show the density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a triangle will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 2, 6))
    else:
        plt.figure(figsize=(n + 2, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n],
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

In [ ]:
def stacked_barplot(data, predictor, target):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 5, 5))
    plt.legend(
        loc="lower left", frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
### function to plot distributions wrt target


def distribution_plot_wrt_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = data[target].unique()

    axs[0, 0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
        stat="density",
    )

    axs[0, 1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
        stat="density",
    )

    axs[1, 0].set_title("Boxplot w.r.t target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()

### Univariate Analysis

In [ ]:
labeled_barplot(data, "no_of_adults",perc=True)

* Around 72% of are of 2 adults, 21% are single adult whereas 6.4% are of 3 Adults

In [ ]:
labeled_barplot(data, "no_of_children",perc=True)

* Around 92.6% of bookings are of zero children and 4.5% are of 1 child and 2.9% are of 2 children

In [ ]:
labeled_barplot(data, "no_of_weekend_nights",perc=True)

* Most of the bookings are of 0 weekend nights and 1 and 2 stands next respectively 

In [ ]:
labeled_barplot(data, "no_of_week_nights",perc=True)

* Most of the booking contains 2 weeknights atleast. next is 1 and 3 and 4 

In [ ]:
labeled_barplot(data, "required_car_parking_space",perc=True)

* Most of the bookings 96.9% doesn't required car parking space.

In [ ]:
histogram_boxplot(data, "lead_time")

* The distribution of `lead_time` is uniformly distributed with some higher values being less frequent.
* Outliers are present in this attribute.

In [ ]:
labeled_barplot(data, "arrival_year",perc=True)

* Most of the bookings 82.0% are of year 2018

In [ ]:
histogram_boxplot(data, "arrival_month")

* Most of the bookings are in Month of October, September, and August.

In [ ]:
histogram_boxplot(data, "arrival_date")

* The distribution of `arrival_date` is uniformly distributed
* No Outliers are present.

In [ ]:
labeled_barplot(data, "repeated_guest",perc=True)

* Most of the Bookings are First timers and and repeating guest are < 3%

In [ ]:
labeled_barplot(data, "no_of_previous_cancellations",perc=True)

* Most of the Bookings 99.1% have not cancelled bookings previously.

In [ ]:
histogram_boxplot(data, "no_of_previous_bookings_not_canceled")

* Most of the Bookings are not cancelled bookings previously.

In [ ]:
histogram_boxplot(data, "avg_price_per_room")

* avg_price_per_room attribute data is normally distributed
* There are outliers in this attribute.

In [ ]:
labeled_barplot(data, "market_segment_type",perc=True)

Most of the bookings are of Online, then Offline.

In [ ]:
labeled_barplot(data, "no_of_special_requests",perc=True)

* Most of the bookings are not having special requests

In [ ]:
labeled_barplot(data, "booking_status",perc=True)

32.8% of bookins are cancelled.

### Bivariate Analysis

In [ ]:
cols_list = data.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(12, 7))
sns.heatmap(
    data[cols_list].corr(), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral"
)
plt.show()

* Lead Time vs. Average Price per Room: There is a moderate positive correlation between lead time and the average price per room. This suggests that bookings made further in advance tend to have higher average room prices.

* Number of Previous Cancellations vs. Lead Time: The number of previous cancellations shows a slightly positive correlation with lead time. This could indicate that guests who tend to cancel their bookings also book further in advance.

* Number of Special Requests vs. Average Price per Room: There is a slight positive correlation between the number of special requests and the average price per room. Higher spending guests might demand more special requests.

* Weekend Nights vs. Week Nights: There's a moderate positive correlation between the number of weekend nights and week nights, which could imply that guests who stay over the weekend are more likely to extend their stay into the weekdays as well.

* Number of Children vs. Number of Adults: The number of children shows a moderate positive correlation with the number of adults, which is expected as larger groups or families tend to book together.

* Required Car Parking Space vs. Number of Special Requests: There is a slight positive correlation between required car parking space and the number of special requests, indicating that guests who need parking may also have more specific needs.

* Repeated Guest: Repeated guest status shows weak correlations with most variables, suggesting that repeated bookings don't heavily influence most of the other variables like special requests or average price per room.

* Lead Time vs. Number of Special Requests: There is a slight positive correlation between lead time and the number of special requests, indicating that guests who plan further ahead are likely to have more specific needs or requests.

In [ ]:
stacked_barplot(data, "no_of_special_requests", "booking_status")

In [ ]:
stacked_barplot(data, "no_of_previous_cancellations", "booking_status")

In [ ]:
distribution_plot_wrt_target(data, "avg_price_per_room", "booking_status")

In [ ]:
stacked_barplot(data, "market_segment_type", "booking_status")

In [ ]:
distribution_plot_wrt_target(data, "lead_time", "booking_status")

In [ ]:
stacked_barplot(data, "room_type_reserved", "booking_status")

In [ ]:
stacked_barplot(data, "repeated_guest", "booking_status")

In [ ]:
stacked_barplot(data, "required_car_parking_space", "booking_status")

In [ ]:
stacked_barplot(data, "type_of_meal_plan", "booking_status")

In [ ]:
stacked_barplot(data, "no_of_week_nights", "booking_status")

In [ ]:
distribution_plot_wrt_target(data, "avg_price_per_room", "market_segment_type")

Avg. Price of Online Market Segment is high compared to other segments.

# **Data Preprocessing**

### Outlier Check

In [ ]:
# outlier detection using boxplot
numeric_columns = data.select_dtypes(include=np.number).columns.tolist()
# dropping release_year as it is a temporal variable

plt.figure(figsize=(15, 12))

for i, variable in enumerate(numeric_columns):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(data[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

### Data Preparation for modelling

In [ ]:
X = data.drop(["booking_status","Booking_ID"], axis=1)
Y = data["booking_status"]

X = pd.get_dummies(X, drop_first=True)

Y = pd.get_dummies(Y, drop_first=True)

X = X.astype(float)

Y = Y.astype(float)

# Splitting data in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42           ## spliting the data into train and test in the ratio 70:30
)

In [ ]:
## Reset the index of y_train so that both x and y have same indexes for training dataset

y_train.reset_index(inplace = True, drop = True)

In [ ]:
print("Shape of Training set : ", X_train.shape)
print("Shape of test set : ", X_test.shape)
print("Shape of Training set : ", y_train.shape)
print("Shape of test set : ", y_test.shape)
print("Percentage of classes in training set:")
print(y_train.value_counts(normalize=True))
print("Percentage of classes in test set:")
print(y_test.value_counts(normalize=True))

### Scaling the Data

In [ ]:
sc = StandardScaler()

X_train_scaled = pd.DataFrame(sc.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(sc.transform(X_test), columns=X_test.columns)

In [ ]:
X_train_scaled.head()

In [ ]:
X_test_scaled.head()

# **Model Building**

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using sklearn
def model_performance_classification(model, predictors, target, threshold = 0.5):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    prob_pred = model.predict(predictors)
    class_pred = [1 if i >= threshold else 0 for i in prob_pred]

    acc = accuracy_score(target, class_pred)  # to compute Accuracy
    recall = recall_score(target, class_pred)  # to compute Recall
    precision = precision_score(target, class_pred)  # to compute Precision
    f1 = f1_score(target, class_pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
def plot_confusion_matrix(model, predictors, target, threshold = 0.5):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    prob_pred = model.predict(predictors)
    class_pred = [1 if i >= threshold else 0 for i in prob_pred]
    cm = confusion_matrix(target, class_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

## Logistic Regression (with Statsmodel)

In [ ]:
# Adding constant to data for Logistic Regression
X_train_with_intercept = SM.add_constant(X_train_scaled)
X_test_with_intercept = SM.add_constant(X_test_scaled)

In [ ]:
LogisticReg = SM.Logit(y_train, X_train_with_intercept).fit()

print(LogisticReg.summary())

### Checking Logistic Regression model performance on training set

In [ ]:
y_pred = LogisticReg.predict(X_train_with_intercept)

In [ ]:
logistic_reg_perf_train = model_performance_classification(
    LogisticReg, X_train_with_intercept, y_train
)
logistic_reg_perf_train

In [ ]:
plot_confusion_matrix(LogisticReg, X_train_with_intercept, y_train)

### Checking Logistic Regression model performance on test set

In [ ]:
logistic_reg_perf_test = model_performance_classification(
    LogisticReg, X_test_with_intercept, y_test
)
logistic_reg_perf_test

In [ ]:
plot_confusion_matrix(LogisticReg, X_test_with_intercept, y_test)

## Naive - Bayes Classifier

In [ ]:
#Build Naive Bayes Model
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

### Checking Naive - Bayes Classifier performance on training set

In [ ]:
nb_perf_train = model_performance_classification(nb_model,X_train_scaled,y_train)  
nb_perf_train

In [ ]:
plot_confusion_matrix(nb_model,X_train_scaled,y_train)

### Checking Naive - Bayes Classifier performance on test set

In [ ]:
nb_perf_test = model_performance_classification(nb_model,X_test_scaled,y_test)  
nb_perf_test

In [ ]:
plot_confusion_matrix(nb_model,X_test_scaled,y_test)

## KNN Classifier (K = 3)

In [ ]:
#Build KNN Model
knn_model = KNeighborsClassifier(n_neighbors =3)  
knn_model.fit(X_train_scaled, y_train)

### Checking KNN Classifier performance on training set

In [ ]:
knn_perf_train = model_performance_classification(knn_model,X_train_scaled,y_train)  
knn_perf_train

In [ ]:
plot_confusion_matrix(knn_model,X_train_scaled,y_train)

### Checking KNN Classifier performance on test set

In [ ]:
knn_perf_test = model_performance_classification(knn_model,X_test_scaled,y_test)  
knn_perf_test

In [ ]:
plot_confusion_matrix(knn_model,X_test_scaled,y_test)

## Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

### Checking Decision Tree Classifier performance on training set

In [ ]:
decision_tree_perf_train = model_performance_classification(dt_model,X_train_scaled,y_train) 
decision_tree_perf_train

In [ ]:
plot_confusion_matrix(dt_model,X_train_scaled,y_train)

### Checking Decision Tree Classifier performance on test set

In [ ]:
decision_tree_perf_test = model_performance_classification(dt_model,X_test_scaled,y_test)  
decision_tree_perf_test

In [ ]:
plot_confusion_matrix(dt_model,X_test_scaled,y_test)

# **Model Performance Improvement**

## Logistic Regression (deal with multicollinearity, remove high p-value variables, determine optimal threshold 	using ROC curve)

### Logistic Regression - Dealing with Multicollinearity

In [ ]:
def calculate_vif(idf):
    """
    Calculate Variance Inflation Factor (VIF) for each variable in a DataFrame.

    Parameters:
    df (DataFrame): Input DataFrame containing numerical variables.

    Returns:
    vif_df (DataFrame): DataFrame containing variable names and their corresponding VIF values.
    """
    variables = idf.values
    vif_df = pd.DataFrame()
    vif_df["Variable"] = idf.columns
    vif_df["VIF"] = [variance_inflation_factor(variables, i) for i in range(idf.shape[1])]
    return vif_df

In [ ]:
# Call the function to calculate VIF
vif_result = calculate_vif(X_train_with_intercept)  

print("Variance Inflation Factors:")
print(vif_result)

In [ ]:
# # Dropping columns with VIF > 5 iteratively
while vif_result['VIF'].max() > 5:
    high_vif_column = vif_result.loc[vif_result['VIF'].idxmax(), 'Variable']
    print(f"Dropping {high_vif_column} due to high VIF")
    X_train_scaled.drop(columns=high_vif_column, inplace=True)
    X_test_scaled.drop(columns=high_vif_column, inplace=True)
    vif_result = calculate_vif(X_train_scaled)

In [ ]:
vif_result

### Dealing with high p-value variables

In [ ]:
# initial list of columns
predictors = X_train_with_intercept.copy()
cols = predictors.columns.tolist()

# setting an initial max p-value
max_p_value = 1

while len(cols) > 0:
    # defining the train set
    x_train_aux = predictors[cols]

    # fitting the model
    model = SM.Logit(y_train, x_train_aux).fit()

    # getting the p-values and the maximum p-value
    p_values = model.pvalues
    max_p_value = max(p_values)

    # name of the variable with maximum p-value
    feature_with_p_max = p_values.idxmax()
    print(f"Dropping column {feature_with_p_max} with p-value: {max_p_value}")

    if max_p_value > 0.05:
        cols.remove(feature_with_p_max)
    else:
        break

selected_features = cols
print(selected_features)

In [ ]:
X_train_significant = X_train_with_intercept[selected_features]
X_test_significant = X_test_with_intercept[selected_features]  
X_train_significant.head(10)

In [ ]:
X_test_significant.head(10)

### Training the Logistic Regression model again with only the significant features

In [ ]:
LogisticReg_tuned = SM.Logit(y_train, X_train_significant).fit() 
print(LogisticReg_tuned.summary())

### Determining optimal threshold using ROC Curve

In [ ]:
y_pred = LogisticReg_tuned.predict(X_train_significant)
fpr, tpr, thresholds = roc_curve(y_train, y_pred)

# Plot ROC curve
roc_auc = roc_auc_score(y_train, y_pred)  ## Complete the code to get the ROC-AUC score
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
# Find the optimal threshold
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold_logit = round(thresholds[optimal_idx], 3)
print("\nOptimal Threshold: ", optimal_threshold_logit)

### Checking tuned Logistic Regression model performance on training set

In [ ]:
logistic_reg_tune_perf_train = model_performance_classification(
    LogisticReg_tuned, X_train_significant, y_train, optimal_threshold_logit
)
logistic_reg_tune_perf_train

In [ ]:
plot_confusion_matrix(LogisticReg_tuned, X_train_significant, y_train, optimal_threshold_logit)

### Checking tuned Logistic Regression model performance on test set

In [ ]:
logistic_reg_tune_perf_test = model_performance_classification(
    LogisticReg_tuned, X_test_significant, y_test, optimal_threshold_logit
)

logistic_reg_tune_perf_test

In [ ]:
plot_confusion_matrix(LogisticReg_tuned, X_test_significant, y_test, optimal_threshold_logit)

## KNN Classifier (different values of K)

In [ ]:
# Define the range for k values
k_values = range(2,21)  #defining the range for k-values between 2 and 20 (both inclusive)

# Initialize variables to store the best k and the highest recall score
best_k = 0
best_recall = 0

# Loop through each k value
for k in k_values:
    # Create and fit the KNN classifier with the current k value
    knn = KNeighborsClassifier(n_neighbors = k)  
    knn.fit(X_train_scaled, y_train)

    # Predict on the test set
    y_pred = knn.predict(X_test_scaled)

    # Calculate the recall score
    recall = recall_score(y_test, y_pred)

    # Print the recall score for the current k value
    print(f'Recall for k={k}: {recall}')

    # Update the best k and best recall score if the current recall is higher
    if recall > best_recall:
        best_recall = recall
        best_k = k

# Print the best k value and its recall score
print(f'\nThe best value of k is: {best_k} with a recall of: {best_recall}')

In [ ]:
knn_tuned = KNeighborsClassifier(n_neighbors =19)  
knn_tuned.fit(X_train_scaled, y_train)

### Checking tuned KNN model performance on training set

In [ ]:
knn_tuned_perf_train = model_performance_classification(knn_tuned,X_train_scaled, y_train) 
knn_tuned_perf_train

In [ ]:
plot_confusion_matrix(knn_tuned,X_train_scaled, y_train)

### Checking tuned KNN model performance on test set

In [ ]:
knn_tuned_perf_test = model_performance_classification(knn_tuned,X_test_scaled, y_test) 
knn_tuned_perf_test

In [ ]:
plot_confusion_matrix(knn_tuned,X_test_scaled, y_test)

## Decision Tree Classifier (pre-pruning)

### Pre-pruning the tree

In [ ]:
# Choose the type of classifier.
dt_model_tuned = DecisionTreeClassifier(random_state=42)

# Grid of parameters to choose from
parameters = {
    "max_depth": np.arange(5, 13, 2),                          ## Max Depth of the decision tree
    "max_leaf_nodes": [10, 20, 40, 50, 75, 100],               ## Maximum number of leaf nodes
    "min_samples_split": [2, 5, 7, 10, 20, 30],                ## Minimum number of samples required to split an internal node
    "class_weight": ['balanced', None]                         ## whether or not to used balanced weights for impurity computations
}

# # Type of scoring used to compare parameter combinations
acc_scorer = make_scorer(recall_score)

# Run the grid search
grid_obj = GridSearchCV(dt_model_tuned, parameters, scoring='recall', cv=5)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
dt_model_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
dt_model_tuned.fit(X_train, y_train)

### Checking tuned Decision Tree Classifier performance on training set

In [ ]:
decision_tree_tuned_perf_train = model_performance_classification(dt_model_tuned,X_train, y_train)  
decision_tree_tuned_perf_train

In [ ]:
plot_confusion_matrix(dt_model_tuned,X_train, y_train)

### Checking tuned Decision Tree Classifier performance on test set

In [ ]:
decision_tree_tuned_perf_test = model_performance_classification(dt_model_tuned,X_test, y_test)  
decision_tree_tuned_perf_test

In [ ]:
plot_confusion_matrix(dt_model_tuned,X_test, y_test)

### Visualizing the Decision Tree

In [ ]:
plt.figure(figsize=(20, 10))
out = tree.plot_tree(
    dt_model_tuned,
    feature_names=X_train.columns.tolist(),
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
# below code will add arrows to the decision tree split if they are missing
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

### Analyzing Feature Importance for tuned Decision Tree Classifier

In [ ]:
# Uncomment and run to check feature importance for Tuned Decision Tree model


# # importance of features in the tree building

feature_names = X_train.columns.tolist()
importances = dt_model_tuned.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8, 8))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

# **Model Performance Comparison and Final Model Selection**

In [ ]:
# training performance comparison

models_train_comp_df = pd.concat(
    [
        logistic_reg_perf_train.T,
        logistic_reg_tune_perf_train.T,
        nb_perf_train.T,
        knn_perf_train.T,
        knn_tuned_perf_train.T,
        decision_tree_perf_train.T,
        decision_tree_tuned_perf_train.T
            ],
    axis=1,
)
models_train_comp_df.columns = [
    "Logistic Regression Base",
    "Logistic Regression Tuned",
    "Naive Bayes Base",
    "KNN Base",
    "KNN Tuned",
    "Decision Tree Base",
    "Decision Tree Tuned"
]
print("Training performance comparison:")
models_train_comp_df

In [ ]:
# testing performance comparison

models_test_comp_df = pd.concat(
    [
        logistic_reg_perf_train.T,
        logistic_reg_tune_perf_train.T,
        nb_perf_test.T,
        knn_perf_test.T,
        knn_tuned_perf_test.T,
        decision_tree_perf_test.T,
        decision_tree_tuned_perf_test.T
    ],
    axis=1,
)
models_test_comp_df.columns = [
    "Logistic Regression Base",
    "Logistic Regression Tuned",
    "Naive Bayes Base",
    "KNN Base",
    "KNN Tuned",
    "Decision Tree Base",
    "Decision Tree Tuned"
]
print("Test set performance comparison:")
models_test_comp_df

# **Actionable Insights and Recommendations**

### Actionable Insights:

Lead Time:

Insight: Longer lead times significantly increase the likelihood of cancellations.

Recommendation: Consider implementing stricter cancellation policies for bookings made well in advance. For example, require a non-refundable deposit for long lead-time bookings.

Market Segment Type (Online):

Insight: Bookings made through online channels have a higher likelihood of cancellation.

Recommendation: Offer incentives such as discounts or loyalty points for customers who book directly through the hotel's website or app. This can help reduce dependency on third-party platforms and potentially lower cancellation rates.

Average Price per Room:

Insight: Higher-priced rooms are more likely to be canceled.

Recommendation: Implement flexible pricing strategies to encourage customers to commit to their reservations. For example, offer special non-refundable rates with discounts for higher-priced rooms.

Number of Special Requests:

Insight: Guests who make multiple special requests are more likely to cancel.

Recommendation: Track and review special requests to identify patterns and trends. Consider reaching out to these guests with personalized communication to confirm their reservations and address their needs.

### Recommendations:
Implement Tiered Cancellation Policies:

Develop tiered cancellation policies based on booking characteristics such as lead time, room type, and booking channel. For example, offer flexible cancellation options for last-minute bookings but enforce stricter policies for long lead-time bookings.

Optimize Distribution Channels:

Analyze the performance of various distribution channels and optimize the mix to reduce cancellations. Prioritize channels that have lower cancellation rates and offer incentives for direct bookings.

Enhance Customer Communication:

Improve pre-stay communication with guests to reduce cancellations. Send reminders, special offers, and personalized messages to keep guests engaged and committed to their reservations.

Offer Incentives for Non-Cancellation:

Introduce loyalty programs or incentives for guests who honor their bookings. Offer discounts, loyalty points, or other rewards for guests who consistently avoid cancellations.

Monitor and Adjust Pricing Strategies:

Use dynamic pricing strategies to adjust room rates based on booking patterns and cancellation likelihood. Offer special non-refundable rates or discounts for guests who are less likely to cancel.

Review and Refine Special Requests Management:

Monitor special requests closely and identify guests with high cancellation risk. Offer personalized services and communication to address their needs and reduce the likelihood of cancellations.

By implementing these insights and recommendations, INN Hotels Group can proactively manage booking cancellations, optimize their revenue, and improve guest satisfaction.